#### Cleaning, Grouping, and Summarizing Data

- checking how many rows and columns are present,
- looking for missing values,
- scanning for obvious problems such as duplicated rows or strange types,
- and only then asking deeper questions.

In [1]:
import pandas as pd

data = {
    "date": [
        "2024-01-01", "2024-01-01", "2024-01-02",
        "2024-01-03", "2024-01-03", "2024-01-03",
    ],
    "channel": [
        "search", "social", "social",
        "search", "email", "social"
    ],
    "signups": [42, 38, None, 55, 17, 55],
}

signups = pd.DataFrame(data)
signups

,date,channel,signups
0,2024-01-01,search,42.0
1,2024-01-01,social,38.0
2,2024-01-02,social,NaN
3,2024-01-03,search,55.0
4,2024-01-03,email,17.0
5,2024-01-03,social,55.0


In [2]:
signups.head()

,date,channel,signups
0,2024-01-01,search,42.0
1,2024-01-01,social,38.0
2,2024-01-02,social,NaN
3,2024-01-03,search,55.0
4,2024-01-03,email,17.0


In [3]:
signups.info()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   date     6 non-null      str    
 1   channel  6 non-null      str    
 2   signups  5 non-null      float64
dtypes: float64(1), str(2)
memory usage: 276.0 bytes


- Notice that there is a NaN value, and the date column contains raw strings and a duplicate.
In pandas, missing or undefined values are represented as `NaN`, which stands for "Not a Number." 
`NaN` is a special floating-point value, so columns with missing integer values are converted to floats (float64) to accommodate it.
If you need the column to be integers (e.g., after handling missing values), remember to convert it back with `.astype('int64')`.

#### Handling missing data

In [4]:
print(signups.isna().sum())

date       0
channel    0
signups    1
dtype: int64


In [5]:
# Deletion (agressive)
droped_signups = signups.dropna(subset=["signups"])

# Fill
filled_signups = signups.fillna(signups["signups"].mean())

print(droped_signups)
print(f"-" * 30)
print(filled_signups)

         date channel  signups
0  2024-01-01  search     42.0
1  2024-01-01  social     38.0
3  2024-01-03  search     55.0
4  2024-01-03   email     17.0
5  2024-01-03  social     55.0
------------------------------
         date channel  signups
0  2024-01-01  search     42.0
1  2024-01-01  social     38.0
2  2024-01-02  social     41.4
3  2024-01-03  search     55.0
4  2024-01-03   email     17.0
5  2024-01-03  social     55.0


#### Converting types

In [8]:
filled_signups["date"] = pd.to_datetime(filled_signups["date"], format="%Y-%m-%d")

filled_signups.dtypes

date       datetime64[us]
channel               str
signups           float64
dtype: object

#### Duplicates

Identify duplicates and decide how to handle them.

In [9]:
filled_signups.duplicated().sum()

np.int64(0)

In [11]:
from re import sub

filled_signups.duplicated(subset=["date", "channel"]).sum()

np.int64(0)

#### Grouping once the table is cleaned

In [ ]:
# How mane signups per channel?

by_channel = (
    filled_signups
    .groupby("channel")["signups"]
    .sum()
    .sort_values(ascending=False)
    )

print(by_channel) # The result is a series

channel
social    134.4
search     97.0
email      17.0
Name: signups, dtype: float64


In [16]:
# Chanel summary with .agg()

channel_summary = (
    filled_signups
    .groupby("channel")["signups"]  # group the data by 'channel' and select the 'signups' column for aggregation
    .agg(total = "sum", average = "mean", count = "size")
    .sort_values(by = "total", ascending = False)
)

print(channel_summary)

         total  average  count
channel                       
social   134.4     44.8      3
search    97.0     48.5      2
email     17.0     17.0      1


#### Adding a feature

weekday or weekend?

In [17]:
filled_signups["weekday_name"] = (
    filled_signups["date"].dt.day_name()
)

print(filled_signups)

        date channel  signups weekday_name
0 2024-01-01  search     42.0       Monday
1 2024-01-01  social     38.0       Monday
2 2024-01-02  social     41.4      Tuesday
3 2024-01-03  search     55.0    Wednesday
4 2024-01-03   email     17.0    Wednesday
5 2024-01-03  social     55.0    Wednesday
